## 1DECISION TREE
```
Objective:
The objective of this assignment is to apply Decision Tree Classification to a given dataset, analyse the performance of the model, and interpret the results.
Tasks:

1. Data Preparation:
Load the dataset into your preferred data analysis environment (e.g., Python with libraries like Pandas and NumPy).

2. Exploratory Data Analysis (EDA):
Perform exploratory data analysis to understand the structure of the dataset.
Check for missing values, outliers, and inconsistencies in the data.
Visualize the distribution of features, including histograms, box plots, and correlation matrices.

3. Feature Engineering:
If necessary, perform feature engineering techniques such as encoding categorical variables, scaling numerical features, or handling missing values.

4. Decision Tree Classification:
Split the dataset into training and testing sets (e.g., using an 80-20 split).
Implement a Decision Tree Classification model using a library like scikit-learn.
Train the model on the training set and evaluate its performance on the testing set using appropriate evaluation metrics (e.g., accuracy, precision, recall, F1-score, ROC-AUC).

5. Hyperparameter Tuning:
Perform hyperparameter tuning to optimize the Decision Tree model. Experiment with different hyperparameters such as maximum depth, minimum samples split, and criterion.

6. Model Evaluation and Analysis:
Analyse the performance of the Decision Tree model using the evaluation metrics obtained.
Visualize the decision tree structure to understand the rules learned by the model and identify important features

Interview Questions:
1. What are some common hyperparameters of decision tree models, and how do they affect the model's performance?
2. What is the difference between the Label encoding and One-hot encoding?

```

## Answers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



xls_file = pd.ExcelFile("heart_disease.xlsx")
print(xls_file.sheet_names)

#Loading our Heart_disease file into the enviroment
dataset = pd.read_excel("heart_disease.xlsx", sheet_name="Heart_disease")

# Working on a copy
df = dataset.copy()

#Undestanding data
print("\n<-----------INFO---------->\n")
print(df.info())

print("\n<------------DESCRIBE----------->\n")
print(df.describe())

print("\n<------------DESCRIBE ALL CATEGORICAL AND NUMERICAL VALUES----------->\n")
print(df.describe(include='all'))

print("\n<------------CHCKING NULL VALUES----------->\n")
print(df.isnull().sum())

## Exploratory Data Analysis (EDA)

In [ ]:
#Extracting numerical and categorical columns
target = 'num'
numerical_cols= df.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = df.select_dtypes(exclude=['number']).columns.tolist()

#Removing num cause it is target category
numerical_cols.remove('num')

print(numerical_cols)
print(categorical_cols)

In [ ]:
# Univariate analysis
for col in numerical_cols:
    plt.figure(figsize=(18,9))

    plt.subplot(1,2,1)
    plt.title(f"HistPlot of {col}")
    sns.histplot(df[col],kde=True,bins=20)

    plt.subplot(1,2,2)
    plt.title(f"Boxplot of {col}")
    sns.boxplot(df[col])

    plt.show()

print("\n<---------Univariate analysis of categorical varibales----------->\n")
for col in categorical_cols:
    plt.figure(figsize=(20,9))
    plt.title(f"Bargraph of {col}")
    sns.countplot(x=df[col])
    plt.xticks(rotation=45)

    plt.show()

In [ ]:
# Crrecting the errors in the columns
correction = {
    'FALSE': False,
    'TURE':True,
}
df['exang'] = df['exang'].replace(correction)
df['exang'].astype(bool)
print(df['exang'].info())

In [ ]:
# Bivariate Aalysis of numeriacl and categoricals varibles

#NUmerical vs target 
for col in numerical_cols:
    plt.figure(figsize=(20,9))
    sns.boxplot(x='num',y=col, data=df)
    plt.xlabel('num')
    plt.ylabel(col)

    plt.show()

In [ ]:
# Relationship betweeen numerical variables
for i,x_col in enumerate(numerical_cols):
    for y_col in numerical_cols[i+1:]:
        plt.figure(figsize=(20,9))
        sns.scatterplot(x=x_col, y = y_col, data= df)
        plt.xlabel(x_col)
        plt.ylabel(y_col)

        plt.show()

In [ ]:
# Categorical vs target

# making counteplot to see the frequency distibution
for col in categorical_cols:
    plt.figure(figsize=(18,9)) 

    plt.subplot(1,2,1)
    plt.title(f"Countplot of {col}")
    sns.countplot(x=col,hue='num', data=df)
    plt.xticks(rotation=45)

    plt.subplot(1,2,2)
    plt.title(f"Barlot of {col}")
    sns.barplot(x=col, y='num', data=df)
    plt.xticks(rotation=45)
    
    plt.show()
    

In [ ]:
# Bivariate analysis between categorical variables and numerical variables.
for x_col in categorical_cols:
    for y_col in numerical_cols:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=x_col,y=y_col, data=df)
        plt.xlabel(x_col)
        plt.ylabel(y_col)

        plt.show()

## Data Cleaning and Character Encoding

In [ ]:
# Null value Treatment
print(df['oldpeak'])
df.fillna({'oldpeak': df['oldpeak'].median()}, inplace=True)

print(df['oldpeak'])

In [ ]:
outlier_summary = {}
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = (df[col] < lower_limit) | (df[col] > upper_limit)
    outlier_summary[col] = outliers.sum()   # count True values

outliers_count = pd.DataFrame.from_dict(outlier_summary, orient='index', columns=['Outlier_Count'])

print("\n<---------No of outliers in each columns---------->\n")
print(outliers_count)

In [ ]:
#As our dataset is not big clipig the outliers would be good enough
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower_limit, upper_limit)

In [ ]:
# CHecking unique values before applying the encoding
for col in categorical_cols:
    print(f"{col} -->",df[col].unique())

In [ ]:
# label Encoding for the two unique values (Binary columns)
from sklearn.preprocessing import LabelEncoder
binary_cols = ['sex','fbs','exang']
le = LabelEncoder()

for col in binary_cols:
    df[col] = le.fit_transform(df[col])
    
# One hot Encoding for the multiclass columns
df = pd.get_dummies(df, columns=['cp', 'restecg', 'slope', 'thal'], drop_first=True)

print(df.head())


In [ ]:
# Converting all the True and False into 0 and 1 
df = df.map(lambda x: 1 if x is True else (0 if x is False else x))

print(df.head())

In [ ]:
# Dividing the dataset into  target and independent variables
X = df[[col for col in df.columns.tolist() if not col == 'num']]
y = df['num']

print("\n<---------Independent Variables------------->\n")
print(X)

print("\n<----------Target Variables---------->\n")
print(y)

## Model Building and Hyperparameter Tunning

In [ ]:
# Train-test split
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.30, random_state=42)

tree_model = DecisionTreeClassifier()

parameters = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None,1,2, 3,5],    
    'min_samples_split': [2, 5, 10, 20],      
    'min_samples_leaf': [1, 2, 4, 10],         
    'max_features': [None, 'sqrt', 'log2']     
}

decision_tree_model = GridSearchCV(tree_model,parameters,scoring='accuracy',cv=5 )
decision_tree_model.fit(X_train,y_train)

In [ ]:
print(decision_tree_model.best_params_)
print(decision_tree_model.best_score_)

In [ ]:
# Evaluating  on test set
y_pred = decision_tree_model.best_estimator_.predict(X_test)
y_pred_train = decision_tree_model.best_estimator_.predict(X_train)

print("Testing training accuracy:",accuracy_score(y_pred_train,y_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Visualizing the tree model
from sklearn import tree
# Get the best decision tree from GridSearchCV
best_tree = decision_tree_model.best_estimator_


plt.figure(figsize=(15,10))
tree.plot_tree(best_tree, 
               filled=True, 
               feature_names=X.columns, 
               class_names=[str(cls) for cls in y.unique()])

plt.show()

```
Interview Questions:
1. What are some common hyperparameters of decision tree models, and how do they affect the model's performance?
2. What is the difference between the Label encoding and One-hot encoding?
```
### Answers
```
1.) What are some common hyperparameters of decision tree models, and how do they affect the model's performance?
Ans:Common hyperparameters of decision tree are:
i. criterion: Decides how to measure split quality.
ii. max_depth: Limits tree_depth.It may prevent overfitting if set small.
iii. min_samples_test: Minimum sample needed to split the node, higher sample makes tree less complex.
iv. min_samples_leaf: controls minimum samples per leaf. It helps in avoiding overfitting and ensures leaves aren't too small.
v. max_features: Number of features considered per split, which adds randomness and improves generalization.


2.What is the difference between the Label encoding and One-hot encoding?
Ans:

Label Encoding:
i.Label encoding assigns integer values like 0,1,2,3,4 to the categories.
ii.Risk is that model may assume ordinal relationship.

One H0t Encoding:
i. creates separate binary columns for each category.
ii. There us no ordinal assumption but incrase dimentionality.

```